## Build tables from the images

In [16]:
import sys
sys.path.append("../")

from pathlib import Path
from feature_engineering.build_tables import collect_dataset_items_tif, build_split_table_tif
from feature_engineering.build_tables_extended import collect_items_extended, build_split_table_tif_extended
from evaluation.generate_test import spatial_train_test_split

from config import ROOT_NAME, TRAIN_FILES, LABELS_PATH, TEST_FILES_TEMP, \
                   FEATURE_COLS_BASELINE, FEATURE_COLS_BASELINE_EXTENDED

PROJECT_ROOT = Path("../").resolve()

ROOT = (PROJECT_ROOT / ROOT_NAME).resolve()
LABELS_PATH = (PROJECT_ROOT / LABELS_PATH).resolve()

In [17]:
# keep just 1 file per each (for time's sakes)
TRAIN_FILES = [TRAIN_FILES[0]]
TEST_FILES_TEMP = [TEST_FILES_TEMP[0]]

TEST_FILES_TEMP = ['urn:eop:VITO:TERRASCOPE_S2_TOC_V2:S2B_20200406T101559_32UPV_TOC_V210__20200406T101559']

In [18]:
TRAIN_FILES, TEST_FILES_TEMP

(['urn:eop:VITO:TERRASCOPE_S2_TOC_V2:S2B_20200327T101629_32UPV_TOC_V210__20200327T101629'],
 ['urn:eop:VITO:TERRASCOPE_S2_TOC_V2:S2B_20200406T101559_32UPV_TOC_V210__20200406T101559'])

In [19]:
# baseline feature engineering pipeline
items = collect_dataset_items_tif(
    root=ROOT,
    folder_names=TRAIN_FILES,
    label_path=LABELS_PATH,

)

df_baseline = build_split_table_tif(
    items=items,
    kernel_size=11,
    keep_borders=True,
    stride=200,
    distribution=True,
)

df_baseline.shape

Processing urn:eop:VITO:TERRASCOPE_S2_TOC_V2:S2B_20200327T101629_32UPV_TOC_V210__20200327T101629 ...


(5353, 41)

In [20]:
# new pipeline (baseline + features from aryan)
items_extended = collect_items_extended(
    root=ROOT,
    folder_names=TRAIN_FILES,
    label_path=LABELS_PATH,
)

df_extended = build_split_table_tif_extended(
            items=items_extended,
            kernel_size=11,
            keep_borders=True,
            stride=200,
            distribution=True,
)

df_extended.shape

Processing urn:eop:VITO:TERRASCOPE_S2_TOC_V2:S2B_20200327T101629_32UPV_TOC_V210__20200327T101629 ...


(5353, 48)

In [25]:
# generate 1 for temporal test (from 2020, but different dates)
items_test = collect_items_extended(
    root=ROOT,
    folder_names=TEST_FILES_TEMP,
    label_path=LABELS_PATH,
)

test_temporal = build_split_table_tif_extended(
            items=items_test,
            kernel_size=11,
            keep_borders=True,
            stride=200,
            distribution=True,
)

test_temporal.shape

Processing urn:eop:VITO:TERRASCOPE_S2_TOC_V2:S2B_20200406T101559_32UPV_TOC_V210__20200406T101559 ...


(5353, 48)

### Make train/test split with 2 different hold-out types

In [26]:
train_coors_df, test_coors_df = spatial_train_test_split(df_extended, "x", "y")
train_df = df_extended.merge(train_coors_df, on=["x", "y"], how="inner")

# same dates, diff xy
test_spatial = df_extended.merge(test_coors_df, on=["x", "y"], how="inner")

# same xy, diff dates
test_temporal_only = test_temporal.merge(train_coors_df, on=["x", "y"], how="inner")

# diff xy, diff dates
test_spatial_temporal = test_temporal.merge(test_coors_df, on=["x", "y"], how="inner")

In [27]:
# even too much for some of them lol
train_df.shape, test_spatial.shape, test_temporal_only.shape, test_spatial_temporal.shape

((4282, 48), (1071, 48), (4282, 48), (1071, 48))

In [28]:
train_df.columns

Index(['image_id', 'label_path', 'timestamp', 'x', 'y', 'x_norm', 'y_norm',
       'year', 'month', 'day', 'day_of_year', 'doy_sin', 'doy_cos',
       'center_b02', 'mean_b02', 'std_b02', 'min_b02', 'max_b02', 'center_b03',
       'mean_b03', 'std_b03', 'min_b03', 'max_b03', 'center_b04', 'mean_b04',
       'std_b04', 'min_b04', 'max_b04', 'center_b08', 'mean_b08', 'std_b08',
       'min_b08', 'max_b08', 'center_b11', 'mean_b11', 'std_b11', 'min_b11',
       'max_b11', 'mean_NIR', 'mean_SWIR', 'mean_NDVI', 'mean_NDBI',
       'std_NDVI', 'cloud_pct', 'urban_prop', 'water_prop', 'vegetation_prop',
       'built_up_pct'],
      dtype='str')

## Train baseline

In [30]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import classification_report, mean_absolute_error, mean_squared_error
import os
import random

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

In [31]:
# it's also in the config
FEATURE_COLS_BASELINE = [
    'x', 'y', 'x_norm', 'y_norm',
    'year', 'month', 'day', 'day_of_year', 'doy_sin', 'doy_cos',
    'center_b02', 'mean_b02', 'std_b02', 'min_b02', 'max_b02', 'center_b03',
    'mean_b03', 'std_b03', 'min_b03', 'max_b03', 'center_b04', 'mean_b04',
    'std_b04', 'min_b04', 'max_b04', 'center_b08', 'mean_b08', 'std_b08',
    'min_b08', 'max_b08', 'center_b11', 'mean_b11', 'std_b11', 'min_b11',
    'max_b11'
]

FEATURE_COLS_BASELINE_EXTENDED = ['x', 'y', 'x_norm', 'y_norm',
       'year', 'month', 'day', 'day_of_year', 'doy_sin', 'doy_cos',
       'center_b02', 'mean_b02', 'std_b02', 'min_b02', 'max_b02', 'center_b03',
       'mean_b03', 'std_b03', 'min_b03', 'max_b03', 'center_b04', 'mean_b04',
       'std_b04', 'min_b04', 'max_b04', 'center_b08', 'mean_b08', 'std_b08',
       'min_b08', 'max_b08', 'center_b11', 'mean_b11', 'std_b11', 'min_b11',
       'max_b11', 'mean_NIR', 'mean_SWIR', 'mean_NDVI', 'mean_NDBI',
       'std_NDVI', 'cloud_pct', 'built_up_pct']

TARGET_COLUMNS = ['urban_prop', 'water_prop', 'vegetation_prop']
CLASS_NAMES = ['urban', 'water', 'vegetation']

In [32]:
def normalize_predictions_to_simplex(y_pred: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """
    Clip predictions to [0, +inf) and renormalize rows to sum to 1.
    """
    y_pred = np.asarray(y_pred, dtype=np.float64)
    y_pred = np.clip(y_pred, 0.0, None)

    row_sums = y_pred.sum(axis=1, keepdims=True)
    zero_rows = row_sums.squeeze(-1) < eps

    # fallback: uniform distribution if model predicted all zeros
    if np.any(zero_rows):
        y_pred[zero_rows] = 1.0
        row_sums = y_pred.sum(axis=1, keepdims=True)

    y_pred = y_pred / row_sums
    return y_pred


def dominant_class_from_distribution(y: np.ndarray) -> np.ndarray:
    """
    Convert 3-probability target/prediction into hard class via argmax.
    """
    return np.argmax(y, axis=1)


def fit_and_predict_distribution_model(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: list[str],
    target_cols: list[str],
    alpha: float = 1.0,
):
    X_train = train_df[feature_cols].copy()
    y_train = train_df[target_cols].to_numpy(dtype=np.float64)

    X_test = test_df[feature_cols].copy()
    y_test = test_df[target_cols].to_numpy(dtype=np.float64)

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("regressor", Ridge(alpha=alpha, random_state=42)),
    ])

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred = normalize_predictions_to_simplex(y_pred)

    return model, y_test, y_pred


def evaluate_distribution_predictions(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    class_names: list[str],
    digits: int = 4,
):
    """
    Returns both regression-style metrics and classification_report
    computed on dominant class (argmax).
    """
    metrics = {
        "mae_macro": float(mean_absolute_error(y_true, y_pred)),
        "rmse_macro": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae_per_target": {
            class_names[i]: float(mean_absolute_error(y_true[:, i], y_pred[:, i]))
            for i in range(len(class_names))
        },
        "rmse_per_target": {
            class_names[i]: float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])))
            for i in range(len(class_names))
        },
    }

    y_true_cls = dominant_class_from_distribution(y_true)
    y_pred_cls = dominant_class_from_distribution(y_pred)

    clf_report_dict = classification_report(
        y_true_cls,
        y_pred_cls,
        target_names=class_names,
        digits=digits,
        output_dict=True,
        zero_division=0,
    )

    clf_report_text = classification_report(
        y_true_cls,
        y_pred_cls,
        target_names=class_names,
        digits=digits,
        zero_division=0,
    )

    return {
        "metrics": metrics,
        "classification_report_dict": clf_report_dict,
        "classification_report_text": clf_report_text,
    }


def run_experiment_suite(
    train_df: pd.DataFrame,
    test_sets: dict[str, pd.DataFrame],
    feature_sets: dict[str, list[str]],
    target_cols: list[str],
    class_names: list[str],
    alpha: float = 1.0,
):
    """
    Train one model per feature set on train_df.
    Evaluate on all provided test sets.
    """
    all_results = []

    for feature_set_name, feature_cols in feature_sets.items():
        print("=" * 100)
        print(f"FEATURE SET: {feature_set_name}")
        print(f"N features: {len(feature_cols)}")

        for test_name, test_df in test_sets.items():
            model, y_true, y_pred = fit_and_predict_distribution_model(
                train_df=train_df,
                test_df=test_df,
                feature_cols=feature_cols,
                target_cols=target_cols,
                alpha=alpha,
            )

            result = evaluate_distribution_predictions(
                y_true=y_true,
                y_pred=y_pred,
                class_names=class_names,
            )

            print("-" * 100)
            print(f"Test set: {test_name}")
            print("Regression metrics:")
            print(result["metrics"])
            print()
            print("classification_report (dominant class via argmax):")
            print(result["classification_report_text"])

            all_results.append({
                "feature_set": feature_set_name,
                "test_set": test_name,
                "n_features": len(feature_cols),
                "mae_macro": result["metrics"]["mae_macro"],
                "rmse_macro": result["metrics"]["rmse_macro"],
                "macro_f1": result["classification_report_dict"]["macro avg"]["f1-score"],
                "weighted_f1": result["classification_report_dict"]["weighted avg"]["f1-score"],
                "accuracy": result["classification_report_dict"]["accuracy"],
            })

    return pd.DataFrame(all_results)

In [38]:
feature_sets = {
    "baseline": FEATURE_COLS_BASELINE,
    "baseline_extended": FEATURE_COLS_BASELINE_EXTENDED,
}

test_sets = {
    "test_spatial": test_spatial,
    "test_temporal_only": test_temporal_only,
    "test_spatial_temporal": test_spatial_temporal,
}

results_df = run_experiment_suite(
    train_df=train_df,
    test_sets=test_sets,
    feature_sets=feature_sets,
    target_cols=TARGET_COLUMNS,
    class_names=CLASS_NAMES,
    alpha=1.0,
)

results_df

FEATURE SET: baseline
N features: 35
----------------------------------------------------------------------------------------------------
Test set: test_spatial
Regression metrics:
{'mae_macro': 0.0644673351066079, 'rmse_macro': 0.12976189779638944, 'mae_per_target': {'urban': 0.07905050856019026, 'water': 0.019820604437944714, 'vegetation': 0.09453089232168868}, 'rmse_per_target': {'urban': 0.14366517773909085, 'water': 0.07109629456513088, 'vegetation': 0.15754391122307915}}

classification_report (dominant class via argmax):
              precision    recall  f1-score   support

       urban     1.0000    0.0702    0.1311        57
       water     0.0000    0.0000    0.0000        10
  vegetation     0.9410    1.0000    0.9696      1004

    accuracy                         0.9412      1071
   macro avg     0.6470    0.3567    0.3669      1071
weighted avg     0.9353    0.9412    0.9159      1071

-------------------------------------------------------------------------------------

,feature_set,test_set,n_features,mae_macro,rmse_macro,macro_f1,weighted_f1,accuracy
0,baseline,test_spatial,35,0.064467,0.129762,0.366909,0.915904,0.941176
1,baseline,test_temporal_only,35,0.059093,0.126817,0.355794,0.917884,0.942317
2,baseline,test_spatial_temporal,35,0.060970,0.128548,0.376094,0.917345,0.941176
3,baseline_extended,test_spatial,42,0.012488,0.050031,0.655823,0.983233,0.987862
4,baseline_extended,test_temporal_only,42,0.007626,0.037005,0.662588,0.990666,0.993461
5,baseline_extended,test_spatial_temporal,42,0.008723,0.043876,0.658837,0.984155,0.988796
